In [2]:
import asyncio
import os
import queue
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import sounddevice as sd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.live import AsyncSession

load_dotenv()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

In [3]:
IN_SAMPLE_RATE = 16_000
OUT_SAMPLE_RATE = 24_000
MAX_SECS = 5

model = "gemini-3.8-live"
config = {"response_modalities": ["AUDIO"]}

In [ ]:
loop = None
in_chunks = asyncio.Queue()
out_chunks = queue.Queue()


def make_in_mic_cb(loop, in_queue: asyncio.Queue):
    def in_mic_cb(indata: np.ndarray, frames_count, time_info, status):
        chunk_bytes = indata.reshape(-1).astype("<i2").tobytes()
        loop.call_soon_threadsafe(in_queue.put_nowait, chunk_bytes)

    return in_mic_cb


async def send_mic(session: AsyncSession, in_mic_cb, in_queue: asyncio.Queue):
    with sd.InputStream(
        samplerate=IN_SAMPLE_RATE,
        blocksize=0,
        channels=1,
        dtype="<i2",
        callback=in_mic_cb,
    ):
        while True:
            chunk = await in_queue.get()
            await session.send_realtime_input(
                audio=types.Blob(
                    data=chunk,
                    mime_type="audio/pcm;rate=16000",
                )
            )


async def recv_audio(session: AsyncSession, loop: asyncio.AbstractEventLoop, out_queue: queue.Queue):
    print("yo from recv_audio")
    async for response in session.receive():
        content = response.server_content
        if content is None:
            continue

        if content.model_turn and content.model_turn.parts:
            for part in content.model_turn.parts:
                if part.inline_data:
                    chunk = part.inline_data.data
                    out_queue.put(chunk)

        if content.input_transcription:
            print(f"User: {content.input_transcription.text}")
        if content.output_transcription:
            print(f"Gemini: {content.output_transcription.text}")


async def play_audio(out_queue: queue.Queue):
    print("yo from play audio")
    start_pos = 0
    curr_chunk: np.ndarray | None = None

    def speaker_cb(outdata: np.ndarray, frames_count, time_info, status):
        nonlocal start_pos, curr_chunk

        written = 0
        outdata[:] = 0

        while written < frames_count:
            if curr_chunk is None or start_pos >= curr_chunk.shape[0]:
                try:
                    chunk_bytes = out_queue.get_nowait()
                    curr_chunk = np.frombuffer(chunk_bytes, dtype="<i2")
                    start_pos = 0
                except queue.Empty:
                    return

            n = min(frames_count - written, len(curr_chunk) - start_pos)  # either what is remaining to write or what is remaining in this chunk
            outdata[written : written + n, 0] = curr_chunk[start_pos : start_pos + n]  # write in teh reamining area

            start_pos += n
            written += n

    with sd.OutputStream(
        samplerate=OUT_SAMPLE_RATE,
        blocksize=0,
        channels=1,
        dtype="<i2",
        callback=speaker_cb,
    ):
        while True:
            await asyncio.Event().wait()


async def main():
    global loop
    loop = asyncio.get_running_loop()
    client = genai.Client(api_key=GEMINI_API_KEY)
    in_mic_cb = make_in_mic_cb(loop, in_chunks)

    async with client.aio.live.connect(model=model, config=config) as session:
        await asyncio.gather(
            send_mic(session, in_mic_cb, in_chunks),
            recv_audio(session, loop, out_chunks),
            play_audio(out_chunks),
        )


await main()

yo from send_mic
yo from recv_audio
yo from play audio
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic data...
[send_mic] sending the stupid chunk...
[send_mic] done sending this chunk..
[in_mic_cb] getting input mic dat

CancelledError: 